In [ ]:
!pip install -q streamlit pyngrok pyjwt bcrypt email-validator

In [ ]:
from google.colab import userdata
import os

JWT_SECRET = userdata.get("JWT_SECRET")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

os.environ["JWT_SECRET"] = JWT_SECRET
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
os.environ["EMAIL_ADDRESS"] = EMAIL_ADDRESS
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD

print("✅ All secrets loaded successfully!")

In [ ]:
!mkdir -p .streamlit

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#7c3aed"
backgroundColor = "#f8fafc"
secondaryBackgroundColor = "#ffffff"
textColor = "#1f2937"
font = "sans serif"

In [ ]:
%%writefile db.py

import sqlite3
import bcrypt

DB_NAME = "secure_portal.db"


# ---------------------------
# Database Connection
# ---------------------------
def get_connection():
    return sqlite3.connect(DB_NAME)


# ---------------------------
# Create Tables
# ---------------------------
def init_db():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()


# ---------------------------
# Password Hashing
# ---------------------------
def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()


def verify_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed.encode())


# ---------------------------
# User Registration
# ---------------------------
def register_user(username, email, password, security_question, security_answer):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("""
        INSERT INTO users (username, email, password, security_question, security_answer)
        VALUES (?, ?, ?, ?, ?)
        """, (
            username,
            email,
            hash_password(password),
            security_question,
            hash_password(security_answer.lower())
        ))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()


# ---------------------------
# Login (accepts Username OR Email)
# ---------------------------
def get_user_by_identifier(identifier):
    """Returns (username, email, password_hash) matching a username or email, else None."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT username, email, password FROM users
        WHERE username = ? OR email = ?
    """, (identifier, identifier))
    row = cur.fetchone()
    conn.close()
    return row


def authenticate_user(identifier, password):
    row = get_user_by_identifier(identifier)
    if row:
        return verify_password(password, row[2])
    return False


# ---------------------------
# Check Username / Email Exists
# ---------------------------
def username_exists(username):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM users WHERE username=?", (username,))
    exists = cur.fetchone()
    conn.close()
    return exists is not None


def email_exists(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM users WHERE email=?", (email,))
    exists = cur.fetchone()
    conn.close()
    return exists is not None


# ---------------------------
# Security Question / Answer
# ---------------------------
def get_security_question(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT security_question FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


def verify_security_answer(email, answer):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT security_answer FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    if row:
        return verify_password(answer.lower(), row[0])
    return False


# ---------------------------
# Update Password
# ---------------------------
def update_password(email, new_password):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("UPDATE users SET password=? WHERE email=?", (hash_password(new_password), email))
    conn.commit()
    conn.close()


# ---------------------------
# Get Username
# ---------------------------
def get_username(email):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username FROM users WHERE email=?", (email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


# ---------------------------
# Admin: list all registered users (never returns passwords)
# ---------------------------
def get_all_users():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT username, email, created_at FROM users ORDER BY created_at DESC")
    rows = cur.fetchall()
    conn.close()
    return rows


def user_count():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    n = cur.fetchone()[0]
    conn.close()
    return n


init_db()


In [ ]:
%%writefile otp.py

import random
import smtplib
import os
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Store OTPs temporarily in memory (email -> otp)
otp_storage = {}

EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD")


def generate_otp():
    """Generate a 6-digit OTP"""
    return str(random.randint(100000, 999999))


def send_otp(email):
    """
    Generate and send OTP to the given email.
    Returns True if email sent successfully.
    """
    otp = generate_otp()
    otp_storage[email] = otp

    subject = "Secure Portal - Password Reset OTP"

    text_body = f"""Hello,

Your One-Time Password (OTP) for resetting your password is: {otp}

This OTP is valid for a single use. If you did not request a password reset, please ignore this email.

Regards,
Secure Portal Authentication
"""

    html_body = f"""
    <div style="font-family:Segoe UI,Arial,sans-serif;max-width:480px;margin:auto;
                border:1px solid #e5e7eb;border-radius:12px;overflow:hidden;">
      <div style="background:linear-gradient(135deg,#4f46e5,#7c3aed);padding:24px;text-align:center;">
        <h2 style="color:#ffffff;margin:0;">Secure Portal</h2>
      </div>
      <div style="padding:28px;background:#ffffff;">
        <p style="color:#111827;font-size:15px;">Hello,</p>
        <p style="color:#111827;font-size:15px;">
          Your One-Time Password (OTP) for resetting your password is:
        </p>
        <div style="text-align:center;margin:24px 0;">
          <span style="display:inline-block;background:#f3f4f6;color:#4f46e5;
                       font-size:28px;font-weight:700;letter-spacing:6px;
                       padding:12px 24px;border-radius:8px;">{otp}</span>
        </div>
        <p style="color:#6b7280;font-size:13px;">
          This OTP is valid for a single use only. If you did not request a
          password reset, you can safely ignore this email.
        </p>
      </div>
      <div style="background:#f9fafb;padding:14px;text-align:center;">
        <span style="color:#9ca3af;font-size:12px;">Secure Portal Authentication Module</span>
      </div>
    </div>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = email
    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, email, msg.as_string())
        server.quit()
        return True
    except Exception as e:
        print("Email Error:", e)
        return False


def verify_otp(email, otp):
    """Verify entered OTP."""
    if email not in otp_storage:
        return False
    if otp_storage[email] == otp:
        del otp_storage[email]
        return True
    return False


In [ ]:
%%writefile app.py
import streamlit as st
import re
import jwt
import datetime
import os

from db import *
from otp import *

# =====================================================
# PAGE CONFIG
# =====================================================
st.set_page_config(
    page_title="Secure Portal",
    page_icon="🔐",
    layout="centered",
    initial_sidebar_state="collapsed"
)

JWT_SECRET = os.environ.get("JWT_SECRET", "dev-secret-change-me")

# Hardcoded admin credentials -> defined directly in code, NOT a signup account,
# never stored in the users table.
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"

SECURITY_QUESTIONS = [
    "What is your pet name?",
    "What is your favourite city?",
    "What is your mother's maiden name?",
]

EMAIL_PATTERN = r"^[A-Za-z]{2,}@[A-Za-z]{2,}\.[A-Za-z]{2,}$"

# =====================================================
# THEME / CSS
# =====================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap');

html, body, [class*="css"]  {
    font-family: 'Poppins', sans-serif;
}

#MainMenu, footer, header {visibility: hidden;}

.stApp {
    background:
        radial-gradient(circle at 15% 10%, rgba(124,58,237,0.10) 0%, rgba(124,58,237,0) 40%),
        radial-gradient(circle at 85% 90%, rgba(79,70,229,0.10) 0%, rgba(79,70,229,0) 40%),
        #f6f7fb;
}

@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(14px); }
    to   { opacity: 1; transform: translateY(0); }
}

/* Center column card */
.auth-card {
    position: relative;
    background: #ffffff;
    padding: 2.5rem 2.2rem 1.9rem 2.2rem;
    border-radius: 22px;
    box-shadow: 0 18px 45px rgba(79, 70, 229, 0.14), 0 2px 8px rgba(0,0,0,0.04);
    border: 1px solid #eef0f6;
    margin-bottom: 1.2rem;
    margin-top: 0.6rem;
    animation: fadeInUp 0.35s ease-out;
    overflow: hidden;
}
.auth-card::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 5px;
    background: linear-gradient(90deg,#4f46e5,#7c3aed,#a855f7);
}

.eyebrow {
    text-align: center;
    text-transform: uppercase;
    letter-spacing: 0.14em;
    font-size: 0.68rem;
    font-weight: 700;
    color: #7c3aed;
    margin-bottom: 0.3rem;
}

.brand-header {
    text-align: center;
    margin-bottom: 0.6rem;
}
.brand-header .icon-badge {
    font-size: 2.1rem;
    background: linear-gradient(135deg,#4f46e5,#7c3aed);
    width: 66px; height: 66px;
    line-height: 66px;
    border-radius: 50%;
    display: inline-block;
    margin-bottom: 0.7rem;
    box-shadow: 0 10px 24px rgba(124,58,237,0.35);
}
.brand-header h1 {
    font-size: 1.55rem;
    font-weight: 800;
    color: #1e1b4b;
    margin: 0;
    letter-spacing: -0.01em;
}
.brand-header p {
    color: #6b7280;
    font-size: 0.92rem;
    margin-top: 0.3rem;
}

/* Buttons: primary = solid gradient CTA, secondary = ghost/outline nav action */
.stButton>button {
    width: 100%;
    border-radius: 10px;
    padding: 0.55rem 1rem;
    font-weight: 600;
    transition: all 0.15s ease-in-out;
}

.stButton>button[kind="primary"] {
    border: none;
    background: linear-gradient(135deg,#4f46e5,#7c3aed);
    color: white;
}
.stButton>button[kind="primary"]:hover {
    transform: translateY(-1px);
    box-shadow: 0 8px 20px rgba(79,70,229,0.4);
    color: white;
}

.stButton>button[kind="secondary"] {
    background: #ffffff;
    border: 1.5px solid #e2e4ec;
    color: #4f46e5;
}
.stButton>button[kind="secondary"]:hover {
    border-color: #7c3aed;
    background: #f5f3ff;
    color: #4f46e5;
}
.stButton>button:active { transform: translateY(0px); }

/* Text inputs */
.stTextInput>div>div>input {
    border-radius: 10px;
    padding: 0.55rem 0.8rem;
    border: 1px solid #e2e4ec;
    background: #fbfbfe;
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
    caret-color: #1f2937 !important;
}
.stTextInput>div>div>input::placeholder {
    color: #9ca3af !important;
    -webkit-text-fill-color: #9ca3af !important;
    opacity: 1 !important;
}
.stTextInput>div>div>input:focus {
    border-color: #7c3aed;
    box-shadow: 0 0 0 2px rgba(124,58,237,0.15);
    background: #ffffff;
}

/* Selectbox / radio */
div[data-baseweb="select"] > div {
    border-radius: 10px;
    border: 1px solid #e2e4ec !important;
    background: #fbfbfe !important;
    color: #1f2937 !important;
}
div[data-baseweb="select"] span {
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
}
/* The searchable dropdown has its own hidden text input for typing/filtering.
   Without an explicit background/caret color it can inherit a dark theme
   default, which makes the box look black and hides the typing cursor. */
div[data-baseweb="select"] input {
    color: #1f2937 !important;
    -webkit-text-fill-color: #1f2937 !important;
    caret-color: #1f2937 !important;
    background: transparent !important;
}
div[data-baseweb="popover"] {
    background: #ffffff !important;
}
div[data-baseweb="popover"] ul,
div[data-baseweb="popover"] li {
    background: #ffffff !important;
    color: #1f2937 !important;
}
div[data-baseweb="popover"] li:hover {
    background: #f5f3ff !important;
}
div[data-baseweb="popover"] li,
div[data-baseweb="popover"] span {
    color: #1f2937 !important;
}
div[role="radiogroup"] label,
div[role="radiogroup"] p {
    color: #1f2937 !important;
}

.badge-row { text-align:center; margin-top: 0.4rem; }
.small-link { text-align:center; color:#9ca3af; font-size:0.8rem; margin-top: 0.7rem;}

.pw-meter-bg {
    width: 100%; height: 6px; border-radius: 4px; background: #e5e7eb; margin-top: 4px;
}
.pw-meter-fill { height: 6px; border-radius: 4px; transition: width 0.2s ease; }

.metric-card {
    background: #ffffff;
    border-radius: 16px;
    padding: 1.2rem 1.3rem;
    border: 1px solid #eef0f6;
    box-shadow: 0 6px 20px rgba(0,0,0,0.05);
    text-align: center;
}
.metric-card .icon { font-size: 1.3rem; margin-bottom: 0.2rem; }
.metric-card .val { font-size:1.7rem; font-weight:800; color:#4f46e5; }
.metric-card .lbl { font-size:0.78rem; color:#6b7280; margin-top:0.1rem; }

/* Dashboard detail rows */
.detail-row {
    display: flex;
    align-items: center;
    gap: 0.8rem;
    background: #f9fafb;
    border: 1px solid #eef0f6;
    border-radius: 12px;
    padding: 0.75rem 1rem;
    margin-bottom: 0.6rem;
}
.detail-row .d-icon {
    font-size: 1.15rem;
    background: linear-gradient(135deg,#4f46e5,#7c3aed);
    color: white;
    width: 34px; height: 34px;
    border-radius: 9px;
    display: flex; align-items:center; justify-content:center;
    flex-shrink: 0;
}
.detail-row .d-text .d-label { font-size: 0.72rem; color:#9ca3af; text-transform:uppercase; letter-spacing:0.05em; }
.detail-row .d-text .d-value { font-size: 0.95rem; color:#1f2937; font-weight:600; }

/* Force readable text colors on the specific elements that need it,
   without a blanket rule that would also override button/badge text colors */
.stApp, p, span, div, label {
    color: #1f2937;
}
h1, h2, h3, h4, h5, h6 { color: #1e1b4b !important; }

/* Any text inside a button must always inherit the button's own color
   (white on primary, purple on secondary) instead of the page default */
.stButton>button, .stButton>button * {
    color: inherit !important;
}
.stButton>button[kind="primary"], .stButton>button[kind="primary"] * {
    color: #ffffff !important;
}
.stButton>button[kind="secondary"], .stButton>button[kind="secondary"] * {
    color: #4f46e5 !important;
}

/* Same protection for our custom colored badges/labels so the blanket
   rule above never dims them */
.eyebrow, .eyebrow * { color: #7c3aed !important; }
.metric-card .val, .metric-card .val * { color: #4f46e5 !important; }
.metric-card .lbl, .metric-card .lbl * { color: #6b7280 !important; }
.detail-row .d-icon, .detail-row .d-icon * { color: #ffffff !important; }
.detail-row .d-label, .detail-row .d-label * { color: #9ca3af !important; }
.detail-row .d-value, .detail-row .d-value * { color: #1f2937 !important; }
.brand-header p, .brand-header p * { color: #6b7280 !important; }
.small-link, .small-link * { color: #9ca3af !important; }

div[data-testid="stWidgetLabel"] label,
div[data-testid="stWidgetLabel"] p {
    color: #374151 !important;
    font-weight: 500 !important;
}

.stCaption, div[data-testid="stCaptionContainer"] p {
    color: #6b7280 !important;
}

div[data-testid="stMarkdownContainer"] p {
    color: #1f2937 !important;
}

table, .stTable, div[data-testid="stTable"] {
    color: #1f2937 !important;
}
thead tr th, tbody tr td {
    color: #1f2937 !important;
}

div[data-testid="stDataFrame"] {
    border-radius: 12px;
    overflow: hidden;
    border: 1px solid #eef0f6;
}
</style>
""", unsafe_allow_html=True)


def card_start():
    st.markdown('<div class="auth-card">', unsafe_allow_html=True)


def card_end():
    st.markdown('</div>', unsafe_allow_html=True)


def brand_header(icon, title, subtitle, eyebrow=None):
    eyebrow_html = f'<div class="eyebrow">{eyebrow}</div>' if eyebrow else ""
    st.markdown(f"""
    {eyebrow_html}
    <div class="brand-header">
        <div class="icon-badge">{icon}</div>
        <h1>{title}</h1>
        <p>{subtitle}</p>
    </div>
    """, unsafe_allow_html=True)


def detail_row(icon, label, value):
    st.markdown(f"""
    <div class="detail-row">
        <div class="d-icon">{icon}</div>
        <div class="d-text">
            <div class="d-label">{label}</div>
            <div class="d-value">{value}</div>
        </div>
    </div>
    """, unsafe_allow_html=True)


def password_strength(password):
    """Returns (label, pct, color) for a simple visual strength meter."""
    if not password:
        return "", 0, "#e5e7eb"
    score = 0
    if len(password) >= 8:
        score += 1
    if re.search("[A-Z]", password):
        score += 1
    if re.search("[a-z]", password):
        score += 1
    if re.search("[0-9]", password):
        score += 1
    if re.search("[^A-Za-z0-9]", password):
        score += 1

    levels = {
        0: ("Very weak", 10, "#ef4444"),
        1: ("Weak", 25, "#ef4444"),
        2: ("Fair", 50, "#f59e0b"),
        3: ("Good", 70, "#f59e0b"),
        4: ("Strong", 90, "#10b981"),
        5: ("Excellent", 100, "#10b981"),
    }
    return levels[score]


def render_password_meter(password):
    label, pct, color = password_strength(password)
    if password:
        st.markdown(f"""
        <div class="pw-meter-bg">
            <div class="pw-meter-fill" style="width:{pct}%; background:{color};"></div>
        </div>
        <div style="font-size:0.78rem; color:{color}; margin-top:2px;">{label}</div>
        """, unsafe_allow_html=True)


def centered(width_ratio=(1, 2.4, 1)):
    return st.columns(width_ratio)[1]


# =====================================================
# SESSION STATE
# =====================================================
defaults = {
    "page": "Login",
    "token": None,
    "reset_email": "",
    "otp_sent": False,
    "verified": False,
    "is_admin": False,
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v


# =====================================================
# JWT HELPERS
# =====================================================
def create_token(payload_extra):
    payload = {
        **payload_extra,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


def valid_email(email):
    return re.match(EMAIL_PATTERN, email)


def valid_password(password):
    if len(password) < 8:
        return False
    if not re.search("[A-Z]", password):
        return False
    if not re.search("[a-z]", password):
        return False
    if not re.search("[0-9]", password):
        return False
    if not re.search("[^A-Za-z0-9]", password):
        return False
    return True


def goto(page):
    st.session_state.page = page
    st.rerun()


# =====================================================
# LOGIN
# =====================================================
if st.session_state.page == "Login":
    with centered():
        card_start()
        brand_header("🔐", "Welcome Back", "Sign in to your Secure Portal account", eyebrow="Secure Access")

        identifier = st.text_input("👤 Username or Email", placeholder="yourname or you@example.com")
        password = st.text_input("🔒 Password", type="password", placeholder="••••••••")

        if st.button("Login", type="primary"):
            if identifier == "" or password == "":
                st.error("All fields are mandatory.")
            elif authenticate_user(identifier, password):
                row = get_user_by_identifier(identifier)
                email = row[1]
                st.session_state.token = create_token({"email": email})
                st.session_state.is_admin = False
                goto("Dashboard")
            else:
                st.error("Invalid username/email or password.")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)

        c1, c2 = st.columns(2)
        with c1:
            if st.button("Create Account", type="secondary"):
                goto("Signup")
        with c2:
            if st.button("Forgot Password", type="secondary"):
                goto("Forgot")

        st.markdown(
            "<div class='small-link'>Portal administrator?</div>",
            unsafe_allow_html=True,
        )
        if st.button("Admin Login", type="secondary", key="go_admin"):
            goto("AdminLogin")

        card_end()

# =====================================================
# SIGNUP
# =====================================================
elif st.session_state.page == "Signup":
    with centered():
        card_start()
        brand_header("📝", "Create Account", "Join the Secure Portal in a minute", eyebrow="Get Started")

        username = st.text_input("👤 Username")
        email = st.text_input("✉️ Email")
        password = st.text_input("🔒 Password", type="password")
        render_password_meter(password)
        confirm = st.text_input("🔒 Confirm Password", type="password")
        question = st.selectbox("🛡️ Security Question", SECURITY_QUESTIONS)
        answer = st.text_input("💬 Security Answer")

        st.caption("Password must have 8+ characters, an uppercase & lowercase letter, a number and a special symbol.")

        if st.button("Create Account", type="primary", key="signup_btn"):
            if "" in [username, email, password, confirm, answer]:
                st.error("All fields are mandatory.")
            elif username_exists(username):
                st.error("Username already exists.")
            elif email_exists(email):
                st.error("Email already exists.")
            elif not valid_email(email):
                st.error("Invalid email format.")
            elif not valid_password(password):
                st.error("Password must contain uppercase, lowercase, number and special character.")
            elif password != confirm:
                st.error("Passwords do not match.")
            else:
                ok = register_user(username, email, password, question, answer)
                if ok:
                    st.success("Registration successful! Please login.")
                    goto("Login")
                else:
                    st.error("Registration failed. Please try again.")

        if st.button("Back to Login", type="secondary", key="signup_back"):
            goto("Login")
        card_end()

# =====================================================
# FORGOT PASSWORD
# =====================================================
elif st.session_state.page == "Forgot":
    with centered():
        card_start()
        brand_header("🔑", "Forgot Password", "Recover access using your security question or OTP", eyebrow="Account Recovery")

        method = st.radio("Choose Recovery Method", ["Security Question", "OTP"], horizontal=True)

        # --------------------------
        # Security Question Method
        # --------------------------
        if method == "Security Question":
            email = st.text_input("✉️ Registered Email", key="sq_email")

            if email:
                if email_exists(email):
                    question = get_security_question(email)
                    st.info(f"❓ {question}")

                    answer = st.text_input("💬 Answer", key="sq_answer")
                    new_password = st.text_input("🔒 New Password", type="password", key="sq_pw")
                    render_password_meter(new_password)
                    confirm = st.text_input("🔒 Confirm Password", type="password", key="sq_cp")

                    if st.button("Reset Password", type="primary", key="sq_reset"):
                        if answer == "" or new_password == "" or confirm == "":
                            st.error("All fields are mandatory.")
                        elif not valid_password(new_password):
                            st.error("Password must contain uppercase, lowercase, number and special character.")
                        elif new_password != confirm:
                            st.error("Passwords do not match.")
                        elif verify_security_answer(email, answer):
                            update_password(email, new_password)
                            st.success("Password updated successfully.")
                            goto("Login")
                        else:
                            st.error("Wrong security answer.")
                else:
                    st.error("Email not registered.")

        # --------------------------
        # OTP Method
        # --------------------------
        else:
            email = st.text_input("✉️ Registered Email", key="otp_email")

            if not st.session_state.otp_sent:
                if st.button("Send OTP", type="primary"):
                    if not email_exists(email):
                        st.error("Email not registered.")
                    elif send_otp(email):
                        st.session_state.otp_sent = True
                        st.session_state.reset_email = email
                        st.rerun()
                    else:
                        st.error("Unable to send OTP.")
            else:
                otp = st.text_input("🔢 Enter OTP")

                if st.button("Verify OTP", type="primary"):
                    if verify_otp(st.session_state.reset_email, otp):
                        st.session_state.verified = True
                        st.rerun()
                    else:
                        st.error("Invalid OTP.")

                if st.session_state.verified:
                    new_password = st.text_input("🔒 New Password", type="password", key="otp_pw")
                    render_password_meter(new_password)
                    confirm = st.text_input("🔒 Confirm Password", type="password", key="otp_cp")

                    if st.button("Update Password", type="primary"):
                        if not valid_password(new_password):
                            st.error("Password does not satisfy policy.")
                        elif new_password != confirm:
                            st.error("Passwords do not match.")
                        else:
                            update_password(st.session_state.reset_email, new_password)
                            st.success("Password updated successfully.")
                            st.session_state.otp_sent = False
                            st.session_state.verified = False
                            goto("Login")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)
        if st.button("Back to Login", type="secondary", key="forgot_back"):
            st.session_state.otp_sent = False
            st.session_state.verified = False
            goto("Login")
        card_end()

# =====================================================
# ADMIN LOGIN  (separate page, hardcoded credentials, no signup account)
# =====================================================
elif st.session_state.page == "AdminLogin":
    with centered():
        card_start()
        brand_header("🛡️", "Admin Login", "Restricted access for portal administrators", eyebrow="Admin Only")

        admin_user = st.text_input("👤 Admin Username")
        admin_pass = st.text_input("🔒 Admin Password", type="password")

        if st.button("Login as Admin", type="primary"):
            if admin_user == "" or admin_pass == "":
                st.error("All fields are mandatory.")
            elif admin_user == ADMIN_USERNAME and admin_pass == ADMIN_PASSWORD:
                st.session_state.token = create_token({"role": "admin", "username": ADMIN_USERNAME})
                st.session_state.is_admin = True
                goto("Dashboard")
            else:
                st.error("Invalid admin credentials.")

        st.markdown("<hr style='margin:1.1rem 0 0.6rem 0;'>", unsafe_allow_html=True)
        if st.button("Back to Login", type="secondary", key="admin_back"):
            goto("Login")
        card_end()

# =====================================================
# DASHBOARD (User or Admin)
# =====================================================
elif st.session_state.page == "Dashboard":
    payload = verify_token(st.session_state.token)

    if payload is None:
        st.error("Session expired. Please login again.")
        st.session_state.token = None
        goto("Login")

    # -----------------------------
    # ADMIN DASHBOARD
    # -----------------------------
    if payload.get("role") == "admin":
        with centered((1, 3.2, 1)):
            card_start()
            brand_header("🛡️", "Admin Dashboard", f"Signed in as {payload.get('username')}", eyebrow="Administrator")

            total = user_count()
            m1, m2 = st.columns(2)
            with m1:
                st.markdown(f"""<div class="metric-card"><div class="icon">👥</div><div class="val">{total}</div>
                            <div class="lbl">Registered Users</div></div>""", unsafe_allow_html=True)
            with m2:
                st.markdown("""<div class="metric-card"><div class="icon">🟢</div><div class="val">Online</div>
                            <div class="lbl">Portal Status</div></div>""", unsafe_allow_html=True)

            st.write("")
            st.subheader("Registered Users")
            users = get_all_users()
            if len(users) == 0:
                st.info("No registered users yet.")
            else:
                data = [{"Username": u[0], "Email": u[1], "Joined": u[2]} for u in users]
                st.dataframe(data, use_container_width=True, hide_index=True)

            st.markdown("<hr>", unsafe_allow_html=True)
            if st.button("Logout", type="secondary", key="admin_logout"):
                st.session_state.token = None
                st.session_state.is_admin = False
                goto("Login")
            card_end()

    # -----------------------------
    # NORMAL USER DASHBOARD
    # -----------------------------
    else:
        email = payload["email"]
        username = get_username(email)

        with centered():
            card_start()
            brand_header("🏠", f"Welcome, {username}!", "You're securely logged in", eyebrow="User Dashboard")

            detail_row("👤", "Username", username)
            detail_row("✉️", "Email", email)

            st.markdown("<hr>", unsafe_allow_html=True)
            st.info("You have full access to your account. Stay secure!")

            if st.button("Logout", type="secondary"):
                st.session_state.token = None
                goto("Login")
            card_end()


In [ ]:
from pyngrok import ngrok
import subprocess
import time
import os

# Authenticate ngrok
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# --- Clean up anything left over from a previous run ---
# 1. Kill any lingering Streamlit process
!pkill -f streamlit

# 2. Disconnect any tunnels pyngrok still thinks are open in THIS session
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception as e:
    print("No existing tunnels to disconnect:", e)

# 3. Kill the ngrok agent process itself (this is the part that actually
#    frees up the free-tier static domain — pkill on streamlit alone does NOT do this)
ngrok.kill()

time.sleep(2)

# Start Streamlit
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"]
)

# Wait for Streamlit to start
time.sleep(8)

# Create public URL (fresh agent, since we killed the old one above)
public_url = ngrok.connect(8501)

print("🌐 Open your app here:")
print(public_url.public_url)